# 20 — CDS Greeks via AD

Exact, model-consistent sensitivities for credit default swaps:
- **CS01** (credit spread sensitivity)
- **IR01** (interest rate sensitivity)
- **Recovery sensitivity**
- **Jacobian** w.r.t. all market inputs
- Cross-gamma (second-order)

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms, plot_heatmap
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.instruments.cds import make_cds
from ql_jax.engines.credit.midpoint import midpoint_cds_npv, cds_fair_spread

NOTIONAL = 1_000_000.0
SPREAD = 0.015    # 150 bp
MATURITY = 5.0
RECOVERY = 0.40
HAZARD = 0.02
DISC_RATE = 0.03

cds = make_cds(NOTIONAL, SPREAD, MATURITY, RECOVERY)

---
## 1. CS01 — Spread Sensitivity

In [ ]:
def npv_fn(hazard, rate):
    disc_fn = lambda t: jnp.exp(-rate * t)
    surv_fn = lambda t: jnp.exp(-hazard * t)
    return midpoint_cds_npv(cds, disc_fn, surv_fn)

# CS01 = dNPV / d(hazard) * 1bp = dNPV/dh * 0.0001
cs01_fn = jax.grad(npv_fn, argnums=0)
cs01 = float(cs01_fn(HAZARD, DISC_RATE)) * 0.0001

# IR01 = dNPV / d(rate) * 1bp
ir01_fn = jax.grad(npv_fn, argnums=1)
ir01 = float(ir01_fn(HAZARD, DISC_RATE)) * 0.0001

print(f"CS01 (per bp of hazard) : {cs01:12.2f}")
print(f"IR01 (per bp of rate)   : {ir01:12.2f}")

---
## 2. Recovery Sensitivity

In [ ]:
def npv_recovery(rec):
    c = make_cds(NOTIONAL, SPREAD, MATURITY, rec)
    disc_fn = lambda t: jnp.exp(-DISC_RATE * t)
    surv_fn = lambda t: jnp.exp(-HAZARD * t)
    return midpoint_cds_npv(c, disc_fn, surv_fn)

drec = float(jax.grad(npv_recovery)(RECOVERY))
print(f"dNPV/dRecovery : {drec:12.2f}")

recs = jnp.linspace(0.1, 0.8, 30)
npvs = [float(npv_recovery(r)) for r in recs]

plt.figure(figsize=(8, 5))
plt.plot(np.array(recs)*100, npvs, 'b-', linewidth=2)
plt.xlabel('Recovery Rate (%)')
plt.ylabel('CDS NPV')
plt.title('CDS NPV vs Recovery Rate')
plt.axhline(0, color='gray', ls='--')
plt.grid(True, alpha=0.3)
plt.show()

---
## 3. Full Jacobian

In [ ]:
def npv_all_params(params):
    hazard, rate, rec = params[0], params[1], params[2]
    c = make_cds(NOTIONAL, SPREAD, MATURITY, rec)
    disc_fn = lambda t: jnp.exp(-rate * t)
    surv_fn = lambda t: jnp.exp(-hazard * t)
    return midpoint_cds_npv(c, disc_fn, surv_fn)

params0 = jnp.array([HAZARD, DISC_RATE, RECOVERY])
grad_vec = jax.grad(npv_all_params)(params0)

print(f"dNPV/d(hazard)   : {float(grad_vec[0]):12.2f}")
print(f"dNPV/d(rate)     : {float(grad_vec[1]):12.2f}")
print(f"dNPV/d(recovery) : {float(grad_vec[2]):12.2f}")

---
## 4. Hessian (Cross-Gammas)

In [ ]:
hess = jax.hessian(npv_all_params)(params0)

labels = ['hazard', 'rate', 'recovery']
plot_heatmap(np.array(hess), labels, labels)
plt.title('CDS NPV Hessian (Cross-Gammas)')
plt.show()

print("\nHessian matrix:")
for i, li in enumerate(labels):
    for j, lj in enumerate(labels):
        print(f"  d²NPV/d({li})d({lj}) = {float(hess[i,j]):12.2f}")

---
## 5. Greek Surfaces

In [ ]:
# CS01 as a function of hazard rate and maturity
hazards = jnp.linspace(0.005, 0.05, 30)
mats = [1.0, 3.0, 5.0, 7.0, 10.0]

fig, ax = plt.subplots(figsize=(10, 6))
for m in mats:
    cs01s = []
    for h in hazards:
        c = make_cds(NOTIONAL, SPREAD, float(m), RECOVERY)
        def npv_h(hh):
            return midpoint_cds_npv(c, lambda t: jnp.exp(-DISC_RATE * t),
                                     lambda t: jnp.exp(-hh * t))
        cs01s.append(float(jax.grad(npv_h)(h)) * 0.0001)
    ax.plot(np.array(hazards)*10000, cs01s, label=f'{m:.0f}Y')

ax.set_xlabel('Hazard Rate (bp)')
ax.set_ylabel('CS01')
ax.set_title('CS01 vs Hazard Rate by Maturity')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 6. Exercises

1. **Fair spread Greeks**: Compute $\partial s_{\text{fair}} / \partial h$ and $\partial s_{\text{fair}} / \partial r$ using AD on `cds_fair_spread`.
2. **Time decay (Theta)**: Approximate $\Theta = \partial V / \partial T$ using AD.
3. **AD vs FD benchmark**: Time AD Greeks vs finite-difference Greeks for a portfolio of 100 CDS.

---
*End of Notebook 20*